# 05 · Do Changes Add Value?

The question (proposal RQ2): **do changes since last year predict a new need for help better than a one-time
snapshot?** This model adds, for every item, how it changed between `t-1` and `t`: an activity getting harder, a new
device, going out less, a new fall, household tasks getting harder, weaker on physical tests, and counts of how many
things got worse.

To answer the question fairly, three logistic regressions are fit with the **same recipe and the same split** as the
baseline (`03_model`) and the tree models (`04_trees`):

| Model | Inputs | Answers |
|---|---|---|
| Status only | 37 status-now items + age, sex, living alone, proxy | the baseline |
| Snapshot | status + extra measures (household tasks, physical tests, chronic conditions ...) | best one-time snapshot |
| **Change** | snapshot + change since last year | the model this notebook is about |

**Change vs snapshot** is the key comparison: the same measures, with and without how they changed. Change vs status
shows the total gain over the baseline.

Show aggregate results only. Clear outputs before committing.

In [ ]:
import sys
from pathlib import Path

for root in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (root / "src" / "willow").exists():
        sys.path.insert(0, str(root / "src"))
        break
for name in list(sys.modules):
    if name == "willow" or name.startswith("willow."):
        del sys.modules[name]

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from willow import models as md, metrics as mt
from willow.data import ROOT, load_panel
from willow.features import analytic_windows, build_windows, feature_kind, feature_label

pd.set_option("display.max_columns", 40, "display.width", 160, "display.precision", 3, "display.max_colwidth", 70)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

OUTCOME = "y_any"
SPLIT_ROUND = 11         # must match 03_model so the test set is identical
MODELS = {"status": "LR status only", "snapshot": "LR snapshot", "change": "LR + change"}

## 1. Same Split As The Baseline

Same people held out, same years, same seed, so every model is scored on exactly the same 2,300 test examples.

In [ ]:
a = analytic_windows(build_windows(load_panel()), OUTCOME)
train, test = md.person_time_split(a, SPLIT_ROUND)
y_test, people = test[OUTCOME].values, test.person_id.values
pd.DataFrame({"train": {"windows": len(train), "people": train.person_id.nunique(), "events": int(train[OUTCOME].sum())},
              "test": {"windows": len(test), "people": test.person_id.nunique(), "events": int(test[OUTCOME].sum())}}).T

## 2. Fit All Three With The Same Recipe

For each feature set: tune the penalty `C` by 5-fold person-grouped cross-validation (same folds for all three), fit
on all training data, Platt-recalibrate on the out-of-fold predictions, predict the test set once.

In [ ]:
fits = {fs: md.fit_logistic_model(train, test, fs, OUTCOME) for fs in MODELS}
thresholds = {fs: mt.alert_threshold(f["p_train"]) for fs, f in fits.items()}

pd.DataFrame({MODELS[fs]: {"features": f["n_features"], "chosen C": f["C"],
                           "cross-validated AUROC": f["cv"]["cv_auroc"].max()} for fs, f in fits.items()}).T

In [ ]:
# tuning curves: the chosen C should sit inside the grid, not at its edge
pd.concat({MODELS[fs]: f["cv"].set_index("C") for fs, f in fits.items()}, axis=1)

## 3. Test Results

Point estimates with 95% CIs (bootstrap over people). Targets: AUROC ≥ 0.75, observed/expected 0.9–1.1.

In [ ]:
results = {fs: mt.bootstrap_ci(y_test, f["p_test"], people, thresholds[fs]) for fs, f in fits.items()}

fmt = lambda r: r.apply(lambda row: f"{row.estimate:.3f} ({row.ci_low:.3f}–{row.ci_high:.3f})", axis=1)
display(pd.DataFrame({MODELS[fs]: fmt(r) for fs, r in results.items()}))
pd.concat({MODELS[fs]: mt.check_targets(r["estimate"].to_dict()).set_index("metric")[["value", "met"]]
           for fs, r in results.items()}, axis=1)

## 4. Do Changes Add Value? (Paired Comparisons)

Each comparison scores both models on the *same* resampled test people, so the CI is for the difference itself.
A CI that stays above 0 means the second model is reliably better on this test set.

In [ ]:
pairs = [("snapshot", "change", "Change vs snapshot (the effect of change features)"),
         ("status", "change", "Change vs status only (total gain over baseline)"),
         ("status", "snapshot", "Snapshot vs status only (effect of extra measures)")]
comparisons = pd.concat({label: mt.paired_difference(y_test, fits[a_]["p_test"], fits[b_]["p_test"], people)
                         for a_, b_, label in pairs})
comparisons

## 5. What The Alerts Mean In People

Per 1,000 at-risk older adults, flagging the top 20% of risk (threshold fixed on training data).

In [ ]:
pd.DataFrame({MODELS[fs]: mt.per_1000(y_test, f["p_test"], thresholds[fs]) for fs, f in fits.items()}).round(0)

## 6. Calibration

Deciles of predicted risk on the test set, baseline vs change model. The diagonal is perfect calibration.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
lim = 0
for fs, color in [("status", "#9a9994"), ("change", "#2a78d6")]:
    cal = mt.calibration_table(y_test, fits[fs]["p_test"])
    ax.plot(cal.predicted, cal.observed, "o-", color=color, ms=6, label=MODELS[fs])
    lim = max(lim, cal.predicted.max(), cal.observed.max())
lim *= 1.1
ax.plot([0, lim], [0, lim], ls="--", color="#bbb", lw=1)
ax.set(xlim=(0, lim), ylim=(0, lim), xlabel="Predicted risk", ylabel="Observed rate",
       title="Calibration on the test set (deciles)")
ax.xaxis.set_major_formatter(lambda v, _: f"{v:.0%}"); ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.legend(frameon=False)
plt.tight_layout()
cal_change = mt.calibration_table(y_test, fits["change"]["p_test"])
cal_change

## 7. What Drives The Change Model

Odds ratios per one standard deviation. With 137 overlapping inputs and strong shrinkage, single coefficients are
small; look at the order and at which *kinds* of input matter.

In [ ]:
coefs = md.coefficients(fits["change"]["model"], fits["change"]["columns"])
coefs.insert(1, "label", coefs.feature.map(feature_label))
coefs.insert(2, "kind", coefs.feature.str.removesuffix(" (missing)").map(feature_kind))
coefs.head(15)

In [ ]:
# the strongest change features on their own
coefs[coefs.kind == "change"].head(10)

In [ ]:
# share of the model's total effect size from status-now vs change inputs
coefs.assign(abs_coef=coefs.coef.abs()).groupby("kind").abs_coef.agg(["count", "sum"]).assign(
    share=lambda d: d["sum"] / d["sum"].sum())

## 8. Where Does The Gain Come From?

The change set adds two different things: **past help** (had help from someone last year, none now; 5% of
windows) and **declines** (activities getting harder, new devices, going out less, more items worse ...). Two more
fits separate them:

- **Snapshot + past help:** the snapshot plus that one history item.
- **Change without past help:** every change feature except past help.

In [ ]:
for fs, label in [("snapshot_history", "LR snapshot + past help"), ("change_no_history", "LR change without past help")]:
    MODELS[fs] = label
    fits[fs] = md.fit_logistic_model(train, test, fs, OUTCOME)

ablation = pd.concat({label: mt.paired_difference(y_test, fits[a_]["p_test"], fits[b_]["p_test"], people) for a_, b_, label in [
    ("snapshot", "snapshot_history", "Past help alone (snapshot + past help vs snapshot)"),
    ("snapshot", "change_no_history", "Declines alone (change without past help vs snapshot)"),
    ("snapshot_history", "change", "Declines on top of past help (change vs snapshot + past help)"),
]})
ablation

## 9. Save Aggregate Outputs

In [ ]:
out = os.path.join(ROOT, "results")
# lr_status is saved by 03_model and lr_change metrics by 04_trees; only what is new here is written
f = fits["snapshot"]
results["snapshot"].assign(model="lr_snapshot", C=f["C"], threshold=thresholds["snapshot"]).to_csv(
    os.path.join(out, "model_lr_snapshot_metrics.csv"))
f["cv"].to_csv(os.path.join(out, "model_lr_snapshot_cv.csv"), index=False)
pd.concat([comparisons, ablation]).to_csv(os.path.join(out, "comparisons_lr.csv"))
coefs.to_csv(os.path.join(out, "model_lr_change_coefficients.csv"), index=False)
fig.savefig(os.path.join(out, "fig_lr_change_calibration.png"), dpi=200, bbox_inches="tight")
print("saved results/model_lr_snapshot_*.csv, comparisons_lr.csv, model_lr_change_coefficients.csv, fig_lr_change_calibration.png")

## Takeaways

- **The change model meets both targets:** AUROC 0.784 (95% CI 0.75–0.81) and observed/expected 0.94. The
  status-only baseline was 0.745.
- **Per 1,000 at-risk adults** (top-20% alerts): 57 correct alerts vs 52 for the baseline, 166 false alarms vs 183,
  and 37 missed vs 41. About 1 in 4 alerts is right (precision 0.25), up from 1 in 5.
- **Calibration improved:** slope 0.94 vs 0.88, so high predicted risks are less inflated.
- **Change vs snapshot is significant** (AUROC +0.018, CI +0.008 to +0.031), but section 8 shows **where it comes
  from: almost all of it is one history item, "had help last year (none now)".** The declines themselves (activities
  getting harder, new devices, going out less) add about +0.003 AUROC, and that's within noise.
- **Why declines add little here:** someone whose activities got harder usually already *looks* worse in their
  current status, so a linear model gets most of that from the snapshot. The univariate signal in `01_explore` is
  real, but it overlaps with current status.
- **What this means:** in logistic regression, "changes add value" is mostly "past help matters". The tree models in
  `04_trees` (random forest 0.787, XGBoost 0.788) are not significantly better than logistic regression with change
  features, so nonlinear interactions don't recover much extra signal either.
- **For Willow:** asking whether someone had help in the past is cheap and adds real signal. The gain from the
  extra measures (snapshot vs status, +0.020) partly relies on physical tests the app can't do at home.